# S48_01 — What Are AI Agents?

An **AI agent** is an LLM that can take actions — calling tools, reading/writing files, browsing the web — and iterate until a goal is achieved. Unlike a single-turn chatbot, agents form a **perception → reasoning → action** loop.

## The agent loop

```
User goal
    ↓
[LLM] → decide action
    ↓
[Tool] → execute action → result
    ↓
[LLM] → observe result → decide next action
    ↓ (repeat until done)
Final answer
```

## Anatomy of an agent

In [ ]:
# Minimal agent loop from scratch — no framework
import anthropic
import json
import math

client = anthropic.Anthropic()

# --- Tools ---
def calculator(expression: str) -> str:
    try:
        result = eval(expression, {'__builtins__': {}}, {'sqrt': math.sqrt, 'pi': math.pi})
        return str(result)
    except Exception as e:
        return f'Error: {e}'

def get_weather(city: str) -> str:
    weather_data = {'London': '15°C, cloudy', 'Paris': '20°C, sunny', 'Tokyo': '22°C, humid'}
    return weather_data.get(city, f'No weather data for {city}')

TOOLS = {
    'calculator': calculator,
    'get_weather': get_weather,
}

# Tool schemas for the API
tool_schemas = [
    {
        'name': 'calculator',
        'description': 'Evaluates a mathematical expression. Use Python syntax.',
        'input_schema': {
            'type': 'object',
            'properties': {'expression': {'type': 'string', 'description': 'Math expression to evaluate'}},
            'required': ['expression'],
        },
    },
    {
        'name': 'get_weather',
        'description': 'Get current weather for a city.',
        'input_schema': {
            'type': 'object',
            'properties': {'city': {'type': 'string', 'description': 'City name'}},
            'required': ['city'],
        },
    },
]

print('Tools defined:', list(TOOLS.keys()))

In [ ]:
def run_agent(user_query, max_steps=5):
    messages = [{'role': 'user', 'content': user_query}]
    
    for step in range(max_steps):
        response = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=1024,
            tools=tool_schemas,
            messages=messages,
        )
        
        # If LLM is done, return final answer
        if response.stop_reason == 'end_turn':
            return response.content[0].text
        
        # Process tool calls
        tool_results = []
        for block in response.content:
            if block.type == 'tool_use':
                tool_fn = TOOLS[block.name]
                result = tool_fn(**block.input)
                print(f'  Step {step+1}: {block.name}({block.input}) → {result}')
                tool_results.append({
                    'type': 'tool_result',
                    'tool_use_id': block.id,
                    'content': result,
                })
        
        # Add assistant turn + tool results to history
        messages.append({'role': 'assistant', 'content': response.content})
        messages.append({'role': 'user', 'content': tool_results})
    
    return 'Max steps reached'

# Run the agent
result = run_agent('What is sqrt(144) + 7? Also, what\'s the weather in Paris?')
print('\nFinal answer:', result)

## Agent components

| Component | Role | Examples |
|-----------|------|----------|
| **LLM brain** | Reasoning, planning, deciding | Claude, GPT-4o, Gemini |
| **Tools** | Interface to the world | Web search, code execution, DB queries |
| **Memory** | Persistent state across turns | Conversation history, vector store, files |
| **Planner** | Break goal into sub-tasks | ReAct, CoT, tree-of-thought |
| **Executor** | Run actions safely | Sandbox, permission checks |

## When to use agents vs RAG vs plain LLM

| Task | Use |
|------|-----|
| Static Q&A over documents | RAG |
| Multi-step task requiring tool calls | Agent |
| Single-turn generation (summarize, classify) | Plain LLM |
| Autonomous goal-seeking (write + test + fix code) | Agent |

Next: [S48_02_function_calling.ipynb](./S48_02_function_calling.ipynb)